# Eksperyment D — wplyw rozdzielczosci wejscia (skala) na transfer

Motywacja (notes/01): synthetic ma DUZE obiekty (mediana bbox ~6776 px2), real MALE (~1180 px2).
Hipoteza HD: wieksza rozdzielczosc wejscia -> male realne obiekty lepiej widoczne -> lepszy transfer,
zwlaszcza AP_small.

Sweep: imgsz 320 / 512(ref) / 768 / 1024. Model YOLOv10n (CNN, najlepszy w transferze),
synthetic 10k, eval cross-domain na realnym holdoucie. ref 512 = eksp A1 (mAP@50 0.459).

GPU A100/H100/Blackwell. ~4 treningi, ~3-5h lacznie.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!git clone https://github.com/JakubPaszke/rareplanes-domain-shift.git 2>/dev/null; cd /content
%cd rareplanes-domain-shift
!git pull -q
!pip -q install ultralytics pycocotools

In [ ]:
# Dane: adnotacje + realny test + synthetic 10k
import os, urllib.request
from concurrent.futures import ThreadPoolExecutor
for d in ['data/real/annotations','data/real/tarballs','data/synthetic/annotations','data/synthetic/images/train']:
    os.makedirs(d, exist_ok=True)
B='https://rareplanes-public.s3.amazonaws.com'
for f in ['instances_train_aircraft.json','instances_test_aircraft.json']:
    !curl -sf -o data/real/annotations/{f} {B}/real/metadata_annotations/{f}
    !curl -sf -o data/synthetic/annotations/{f} {B}/synthetic/metadata_annotations/{f}
!curl -sf -o data/real/tarballs/test.tar.gz {B}/real/tarballs/test/RarePlanes_test_PS-RGB_tiled.tar.gz
!mkdir -p data/real/PS-RGB_tiled && tar -xzf data/real/tarballs/test.tar.gz -C data/real/PS-RGB_tiled/
DST='data/synthetic/images/train'
files=[l.strip() for l in open('configs/synthetic_10k_train_files.txt')]+[l.strip() for l in open('configs/synthetic_10k_val_files.txt')]
files=[f for f in files if f]
def fetch(fn):
    dst=f'{DST}/{fn}'
    if os.path.exists(dst) and os.path.getsize(dst)>0: return True
    try:
        urllib.request.urlretrieve(f'{B}/synthetic/train/images/{fn}', dst+'.part'); os.rename(dst+'.part',dst); return True
    except Exception:
        if os.path.exists(dst+'.part'): os.remove(dst+'.part')
        return False
with ThreadPoolExecutor(max_workers=32) as ex: ok=sum(ex.map(fetch, files))
have=len([f for f in os.listdir(DST) if f.endswith('.png')])
print(f'pobrano {ok}/{len(files)}, na dysku {have}'); assert have>len(files)*0.99
!python3 src/coco_to_yolo.py --domain synthetic --classes aircraft --val-frac 0.15

In [ ]:
# Sweep skali — trening + eval w TEJ SAMEJ rozdzielczosci. workers=8 (natywny Linux OK).
import subprocess
# batche dobrane pod H100/Blackwell ~90GB (YOLOv10n jest maly); umiarkowane,
# zeby nie wymagac zmiany lr (porownywalnosc z eksp A1 batch16). workers 12.
VARIANTS = [(320,128),(768,48),(1024,24)]  # (imgsz, batch); 512 juz mamy z eksp A1
for sz,bs in VARIANTS:
    name=f'expD_{sz}_10k'
    print(f'\n===== {name} (imgsz={sz} batch={bs}) =====')
    subprocess.run(['python3','src/train_yolo.py','--data','data/yolo/synthetic_aircraft/data.yaml',
        '--name',name,'--epochs','45','--batch',str(bs),'--imgsz',str(sz),
        '--seed','42','--patience','15','--workers','12'], check=True)
    subprocess.run(['python3','src/eval_per_size.py','--weights',f'runs/{name}/weights/best.pt',
        '--img-dir','data/real/PS-RGB_tiled/PS-RGB_tiled',
        '--coco-gt','data/real/annotations/instances_test_aircraft.json',
        '--name',f'{name}_ml','--imgsz',str(sz)], check=True)

In [ ]:
# Podsumowanie krzywej skali + AUTO-DOWNLOAD
import json
print('imgsz | mAP@50 | AP_small | AP_large')
print(f'  512 |  0.459 |  0.306   |  0.091   (ref, eksp A1)')
for sz in [320,768,1024]:
    try:
        m=json.load(open(f'results/per_size/expD_{sz}_10k_ml.json'))['metrics']
        print(f' {sz:4d} |  {m["AP@.5"]:.3f} |  {m["AP_small"]:.3f}   |  {m["AP_large"]:.3f}')
    except FileNotFoundError: print(f' {sz:4d} |  (brak)')
from google.colab import files
for sz in [320,768,1024]:
    p=f'results/per_size/expD_{sz}_10k_ml.json'
    if os.path.exists(p): files.download(p)

## Interpretacja
Jesli AP_small rosnie z imgsz (320 < 512 < 768 < 1024) -> hipoteza HD potwierdzona:
wieksza skala pomaga widziec male realne obiekty mimo treningu na duzych syntetycznych.
Plateau / spadek przy 1024 -> jest optymalna skala (koszt obliczeniowy vs zysk).

Ref: imgsz 512 (kafle natywne) = eksp A1 mAP@50 0.459, AP_small 0.306.